# 01 - Fundamentos de Ensambles y Reducción de Varianza

## Curso de Machine Learning: Modelos de Clasificación

En lugar de depender de las predicciones de un único modelo individual, los **Métodos de Ensamble (Ensemble Methods)** combinan las predicciones de múltiples modelos base para construir un estimador global con mayor precisión, robustez y capacidad de generalización.

Este cuaderno sienta las bases matemáticas y conceptuales de por qué los ensambles funcionan, partiendo del problema fundamental del sobreajuste y el dilema sesgo-varianza.

---

### Contenidos
1. El Dilema Sesgo-Varianza: El Problema de los Árboles Individuales
2. Filosofía de los Ensambles y el Teorema del Jurado de Condorcet
3. Bagging (Bootstrap Aggregating) y Reducción de Varianza
4. Matemáticas del Muestreo Out-Of-Bag (OOB)
5. Clasificadores de Votación (`VotingClassifier`): Ensambles Heterogéneos
6. Conclusiones y Transición a Random Forest


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import binom
from sklearn.datasets import load_breast_cancer, make_moons
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)
print("Bibliotecas y entorno listos.")


## 1. El Dilema Sesgo-Varianza: El Problema de los Árboles Individuales

El error de predicción de cualquier modelo de Machine Learning se puede descomponer en tres componentes fundamentales:

$$ \text{Error Total} = \text{Sesgo}^2 + \text{Varianza} + \text{Error Irreducible} $$

*   **Sesgo (Bias):** Errores debidos a suposiciones erróneas en el algoritmo (ej. asumir una relación lineal cuando es cuadrática). Un alto sesgo causa *Subajuste (Underfitting)*.
*   **Varianza (Variance):** Sensibilidad del modelo a las pequeñas fluctuaciones en el conjunto de entrenamiento. Una alta varianza causa *Sobreajuste (Overfitting)*.

### El caso de los Árboles de Decisión
Los árboles de decisión profundos son modelos de **bajo sesgo** (pueden aprender casi cualquier patrón complejo) pero de **muy alta varianza** (si cambias un par de observaciones en el entrenamiento, la estructura del árbol cambia drásticamente).

**La solución de los Ensambles:** Si logramos entrenar múltiples modelos de alta varianza de forma *independiente* y promediamos sus resultados, el sesgo se mantendrá bajo, pero **la varianza se reducirá drásticamente**.


## 2. Filosofía de los Ensambles: La Sabiduría de las Masas

El fundamento probabilístico de por qué promediar modelos funciona proviene del **Teorema del Jurado de Condorcet**.

Si tenemos $M$ clasificadores (jueces) completamente **independientes**, cada uno con una probabilidad individual de acierto $p > 0.5$ (es decir, son ligeramente mejores que lanzar una moneda), la probabilidad de que la votación mayoritaria sea correcta tiende a $1$ a medida que $M \to \infty$.

La probabilidad de que el ensamble acierte por mayoría se calcula con la distribución binomial:

$$\mathbb{P}(\text{Acierto Mayoritario}) = \sum_{k=\lceil M/2 \rceil}^M \binom{M}{k} p^k (1-p)^{M-k}$$

### La Condición Indispensable: Diversidad
Para que este teorema aplique en Machine Learning, los modelos **deben cometer errores diferentes**. Si todos los modelos están altamente correlacionados (cometen los mismos errores en los mismos datos), el ensamble no aportará ninguna mejora sobre un modelo individual.


In [ ]:
# Simulación del Teorema de Condorcet
p_individual = 0.60 # Cada clasificador base tiene solo un 60% de acierto
m_values = np.arange(1, 101, 2) # Cantidad impar de clasificadores

p_mayoritaria = [1.0 - binom.cdf(m // 2, m, p_individual) for m in m_values]

plt.figure(figsize=(10, 5))
plt.plot(m_values, p_mayoritaria, color='#2b5c8f', lw=3, label='Probabilidad Votación Mayoritaria')
plt.axhline(p_individual, color='red', linestyle='--', label=f'Clasificador Individual (p={p_individual})')
plt.axhline(1.0, color='gray', linestyle=':')
plt.xlabel('Número de Clasificadores en el Ensamble (M)', fontweight='bold')
plt.ylabel('Probabilidad de Acierto Global', fontweight='bold')
plt.title('Teorema de Condorcet: Cómo clasificadores débiles crean un ensamble fuerte', fontweight='bold', fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()


## 3. Bagging (Bootstrap Aggregating) y Reducción de Varianza

**Bagging** es el algoritmo clásico para construir ensambles homogéneos (mismo tipo de modelo base, usualmente árboles) forzando la diversidad a través de los datos.

**El Proceso de Bagging:**
1. **Bootstrap:** Dado un dataset de tamaño $N$, generamos $B$ subconjuntos de datos realizando **muestreo aleatorio uniforme con reemplazo** de tamaño $N$.
2. **Entrenamiento Paralelo:** Se entrena un árbol de decisión profundo (sobreajustado intencionalmente) en cada subconjunto.
3. **Agregación:** Para predecir, promediamos las probabilidades de los $B$ árboles (*Soft Voting*) o tomamos la clase mayoritaria (*Hard Voting*).

### La Matemática de la Reducción de Varianza
Si combinamos $B$ variables aleatorias idénticamente distribuidas, con varianza $\sigma^2$ y correlación par a par $\rho$, la varianza de su promedio es:

$$\mathbb{V}ar(\text{Ensamble}) = \rho \sigma^2 + \frac{1 - \rho}{B} \sigma^2$$

El muestreo Bootstrap perturba los datos lo suficiente para reducir la correlación $\rho$ entre los árboles. A medida que incrementamos el número de árboles ($B \to \infty$), el segundo término $\frac{1 - \rho}{B} \sigma^2$ desaparece. **Bagging reduce la varianza drásticamente sin aumentar el sesgo.**


In [ ]:
# Demostración de reducción de sobreajuste con BaggingClassifier
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# 1. Árbol individual (Propenso al sobreajuste)
single_tree = DecisionTreeClassifier(random_state=42)
single_tree.fit(X_train, y_train)
print(f"[Árbol Individual] Exactitud en Train: {single_tree.score(X_train, y_train)*100:.2f}% (Sobreajuste claro)")
print(f"[Árbol Individual] Exactitud en Test:  {single_tree.score(X_test, y_test)*100:.2f}%")

# 2. Ensamble Bagging con 100 árboles
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    oob_score=True, # Veremos esto en la siguiente sección
    random_state=42,
    n_jobs=-1
)
bagging.fit(X_train, y_train)
print(f"\n[Bagging 100 Árboles] Exactitud en Train: {bagging.score(X_train, y_train)*100:.2f}%")
print(f"[Bagging 100 Árboles] Exactitud en Test:  {bagging.score(X_test, y_test)*100:.2f}%")


## 4. Matemáticas del Muestreo Out-Of-Bag (OOB)

Una ventaja colateral maravillosa del muestreo con reemplazo (Bootstrap) es que **obtenemos un conjunto de validación gratuito**.

**Derivación Matemática:**
Si un dataset original tiene $N$ filas, la probabilidad de seleccionar una fila específica en una extracción es $\frac{1}{N}$.
La probabilidad de **NO** seleccionarla es $\left(1 - \frac{1}{N}\right)$.
Como hacemos $N$ extracciones independientes para formar una réplica bootstrap, la probabilidad de que esa fila **nunca** sea seleccionada es:

$$\mathbb{P}(\text{Fila OOB}) = \left(1 - \frac{1}{N}\right)^N$$

Por cálculo de límites, sabemos que a medida que $N$ crece:
$$\lim_{N \to \infty} \left(1 - \frac{1}{N}\right)^N = \frac{1}{e} \approx 0.3678$$

Esto significa que **cada árbol se entrena solo con el ~63.2% de los datos únicos**. El ~36.8% restante (las muestras Out-Of-Bag) son "invisibles" para ese árbol y pueden usarse para evaluar su rendimiento, eliminando la necesidad de recurrir a técnicas costosas como la Validación Cruzada en ensambles masivos.


In [ ]:
oob_score = bagging.oob_score_
test_score = bagging.score(X_test, y_test)

print(f"Estimación Interna del Ensamble (OOB Score): {oob_score*100:.2f}%")
print(f"Evaluación Real en Conjunto de Prueba (Test): {test_score*100:.2f}%")
print("\n*Nota cómo el OOB Score es una estimación increíblemente precisa del rendimiento futuro sin haber tocado X_test.*")


## 5. Clasificadores de Votación (`VotingClassifier`): Ensambles Heterogéneos

Mientras que Bagging promedia el mismo algoritmo (árboles) con distintos datos, el ensamble por votación combina **distintos algoritmos** con los mismos datos. Al usar modelos de naturaleza matemática divergente (espacial, probabilística, basada en distancias), garantizamos que sus errores no estén correlacionados.

- **Voto Duro (Hard Voting):** Cuenta los "votos" discretos de cada clasificador. La clase con más votos gana.
- **Voto Blando (Soft Voting):** Suma las probabilidades continuas $\hat{P}(Y=c \mid \mathbf{x})$ otorgadas por todos los clasificadores a cada clase, y escoge la que tenga mayor probabilidad media. Otorga más peso a los modelos que están más seguros de su predicción y **suele generar mejores resultados**.


In [ ]:
# Creación de modelos algorítmicamente heterogéneos dentro de pipelines (para evitar fuga de datos)
clf1 = Pipeline([('scaler', StandardScaler()), ('logreg', LogisticRegression(random_state=42))])
clf2 = Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=5))])
clf3 = DecisionTreeClassifier(max_depth=4, random_state=42)

# Ensamble por votación suave (Soft Voting)
voting_clf = VotingClassifier(
    estimators=[('lr', clf1), ('knn', clf2), ('dt', clf3)],
    voting='soft'
)

# Evaluación comparativa con Validación Cruzada (5-Fold CV)
modelos = {'Regresión Logística': clf1, 'KNN': clf2, 'Árbol de Decisión': clf3, 'Ensamble Voting': voting_clf}

print("Desempeño Global Evaluado con Validación Cruzada (5-Fold CV):")
print("-" * 60)
for nombre, mod in modelos.items():
    scores = cross_val_score(mod, X_train, y_train, cv=5, scoring='accuracy')
    print(f"{nombre:<22}: {scores.mean()*100:.2f}% (+/- {scores.std()*100:.2f}%)")


## 6. Conclusiones y Transición a Random Forest

1. **El poder de la diversidad:** Los ensambles solo funcionan si los modelos subyacentes son diversos y cometen errores diferentes.
2. **Bagging reduce Varianza:** Al perturbar los datos con muestreo Bootstrap, creamos árboles descorrelacionados que, al promediarse, reducen la varianza y el sobreajuste.
3. **Eficiencia OOB:** Las muestras Bootstrap dejan $\sim36.8\%$ de los datos fuera en cada iteración, permitiendo evaluar el modelo sobre la marcha.

### El Límite de Bagging...
Bagging es poderoso, pero tiene un cuello de botella matemático: si el dataset tiene una variable predictora *demasiado* fuerte, **todos** los árboles bootstrap elegirán esa misma variable para hacer su primer corte. Como resultado, los árboles seguirán estando **muy correlacionados** (el término $\rho\sigma^2$ de la fórmula de varianza no bajará lo suficiente).

En el **Notebook 02**, estudiaremos cómo Leo Breiman resolvió este último obstáculo introduciendo una segunda capa de aleatoriedad para crear el legendario algoritmo **Random Forest**.
